# GAN Example (PyTorch) — Generating Handwritten Digits (MNIST)

The same **DCGAN** as `gan_mnist.ipynb`, implemented in **PyTorch**. Read that notebook's
intro for the full theory; here is the short version plus the PyTorch-specific details.

## Recap of the idea

Two networks play a game:

- **Generator** $G$ maps a random noise vector to a fake `28x28` image.
- **Discriminator** $D$ scores an image as real or fake.

$D$ is trained to catch fakes; $G$ is trained to fool $D$. They co-evolve, and at
convergence $G$ produces images indistinguishable from real MNIST.

## TensorFlow → PyTorch translation, at a glance

| Concept | TensorFlow / Keras | PyTorch |
|---|---|---|
| Define a model | `tf.keras.Sequential([...])` | subclass `nn.Module`, write `forward()` |
| Auto-diff | `tf.GradientTape` | `loss.backward()` |
| Apply gradients | `optimizer.apply_gradients(...)` | `optimizer.step()` |
| Reset gradients | (implicit) | `optimizer.zero_grad()` (PyTorch **accumulates** grads) |
| Train/eval mode | `training=` flag per call | `model.train()` / `model.eval()` |
| Device placement | automatic | explicit `.to(device)` |

The single most common PyTorch beginner bug is forgetting `zero_grad()` — without it,
gradients from previous steps pile up.

> Needs `torch` and `torchvision` (`pip install torch torchvision`). Uses CUDA if available.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch', torch.__version__, '| device:', device)

## 1. Load and prepare the data

`torchvision` downloads MNIST and applies a **transform pipeline**:

- `ToTensor()` converts a PIL image to a tensor and scales pixels to `[0, 1]`.
- `Normalize((0.5,), (0.5,))` then maps them to `[-1, 1]` via `(x - 0.5) / 0.5` — matching
  the generator's `tanh` output, exactly as in the TF version.

A `DataLoader` handles shuffling and batching. `drop_last=True` discards the final
partial batch so every batch has the same size — convenient because batch-norm and our
label tensors assume a fixed batch size.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                 # -> [0, 1]
    transforms.Normalize((0.5,), (0.5,)),  # -> [-1, 1]
])

BATCH_SIZE = 256
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print('Training images:', len(train_dataset))

## 2. The Generator

Same architecture as the Keras version, written as an `nn.Module`. The flow is
`noise → Linear → reshape to 7x7x256 → ConvTranspose2d ×3 → tanh`, doubling the spatial
size at each strided transpose conv (`7→14→28`).

PyTorch transposed-conv arithmetic is explicit: to double a dimension with a stride-2
kernel you typically need `padding` **and** `output_padding`. Here `kernel=5, stride=2,
padding=2, output_padding=1` turns `7×7` into `14×14` and then `14×14` into `28×28`.
We use `BatchNorm`, `LeakyReLU(0.2)`, and a final `Tanh`, mirroring the TF model.

In [ ]:
NOISE_DIM = 100

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(NOISE_DIM, 7 * 7 * 256, bias=False),
            nn.BatchNorm1d(7 * 7 * 256),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 5, stride=1, padding=2, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(128, 64, 5, stride=2, padding=2, output_padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.ConvTranspose2d(64, 1, 5, stride=2, padding=2, output_padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z).view(-1, 256, 7, 7)
        return self.deconv(x)

generator = Generator().to(device)
print(generator)

## 3. The Discriminator

A small CNN that halves the resolution twice (`28→14→7`) then flattens to one logit.
`Dropout(0.3)` keeps it from overpowering the generator. As in the TF version there is
**no final sigmoid** — we output a raw logit and use `BCEWithLogitsLoss` below.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Conv2d(64, 128, 5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.3),

            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
        )

    def forward(self, x):
        return self.net(x)

discriminator = Discriminator().to(device)
print(discriminator)

## 4. Losses and optimizers

`BCEWithLogitsLoss` fuses a sigmoid with binary cross-entropy in one numerically-stable
op, so the networks can emit raw logits. Each network gets its own Adam optimizer with
matched learning rates (`1e-4`) to keep the game balanced — identical to the TF setup.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
g_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4)
d_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4)

In [ ]:
# A fixed noise seed so we can watch the SAME samples improve over time
num_examples = 16
seed = torch.randn(num_examples, NOISE_DIM, device=device)

def plot_samples(noise, epoch=None):
    generator.eval()                       # disable dropout/batch-norm updates
    with torch.no_grad():                  # no gradients needed just to draw
        imgs = generator(noise).cpu() * 0.5 + 0.5  # back to [0, 1]
    generator.train()                      # restore training mode
    grid = make_grid(imgs, nrow=4).permute(1, 2, 0).numpy()
    plt.figure(figsize=(4, 4))
    plt.imshow(grid, cmap='gray')
    plt.axis('off')
    if epoch is not None:
        plt.title(f'Epoch {epoch}')
    plt.show()

## 5. Train

Each batch performs **two updates**, in this order:

1. **Discriminator update.** Generate fakes, score reals and fakes. Note
   `fake.detach()` — this stops gradients from flowing into the generator while we train
   $D$, so the discriminator update only touches discriminator weights.
2. **Generator update.** Re-score the *same* fakes (this time **without** detaching) and
   push them toward the 'real' label. Gradients now flow back through $D$ into $G$.

Each update is the canonical PyTorch trio: `zero_grad()` → `backward()` → `step()`.
`EPOCHS = 30` gives recognisable digits; expect the loss numbers to oscillate.

In [ ]:
EPOCHS = 30

for epoch in range(1, EPOCHS + 1):
    for real, _ in train_loader:
        real = real.to(device)
        bsz = real.size(0)
        ones = torch.ones(bsz, 1, device=device)
        zeros = torch.zeros(bsz, 1, device=device)

        # --- Train discriminator ---
        noise = torch.randn(bsz, NOISE_DIM, device=device)
        fake = generator(noise)
        d_real = discriminator(real)
        d_fake = discriminator(fake.detach())   # detach: don't train G here
        d_loss = criterion(d_real, ones) + criterion(d_fake, zeros)

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # --- Train generator ---
        d_fake_for_g = discriminator(fake)       # no detach: gradient flows into G
        g_loss = criterion(d_fake_for_g, ones)   # G wants D to say 'real'

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(f'Epoch {epoch:3d}/{EPOCHS}  gen_loss={g_loss.item():.3f}  disc_loss={d_loss.item():.3f}')
    if epoch % 5 == 0 or epoch == 1:
        plot_samples(seed, epoch)

## 6. Final generated digits

Feed fresh noise to confirm $G$ generates varied, novel digits.

In [ ]:
plot_samples(torch.randn(num_examples, NOISE_DIM, device=device))

### Things to try

- Increase `EPOCHS` and watch the digits sharpen.
- This code already runs on GPU automatically via the `device` setup.
- Save the generator with `torch.save(generator.state_dict(), 'gan_generator.pt')`.
- Try **label smoothing**: use `0.9` instead of `1.0` for the real label — a classic
  trick that stabilises GAN training.